In [ ]:
!pip install pandas

In [ ]:
!pip install numpy

In [ ]:
!pip install seaborn

In [ ]:
!pip install scikit-learn

In [ ]:
#prepartion DATA
import pandas as pd
data=pd.read_csv('Telco_Cusomer_Churn.csv')
data.head()

In [ ]:
data.isnull().sum()

In [ ]:
data['customerID']

In [ ]:
data.info()

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC


In [ ]:
# Drop customerID if present
if 'customerID' in data.columns:
    data = data.drop('customerID', axis=1)

# Fill missing values
data = data.fillna(method='ffill')

# Encode categorical variables
for col in data.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])

# Separate features and target
X = data.drop('Churn', axis=1)
y = data['Churn']

# Scale numerical features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
scaler

In [ ]:
X_scaled

In [ ]:
data.shape

In [ ]:
#split Data for train and test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
X_train=abs(X_train)


In [ ]:
#feature selection
selector = SelectKBest(score_func=chi2, k=10)
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)


In [ ]:
#model selction

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "SVM": SVC(kernel='rbf', probability=True)
}

for name, model in models.items():
    model.fit(X_train_selected, y_train)
    y_pred = model.predict(X_test_selected)
    print(f"\n{name} Results:")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


In [ ]:
#model training
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(RandomForestClassifier(), param_grid, cv=5, scoring='accuracy')
grid.fit(X_train_selected, y_train)

print("Best Parameters:", grid.best_params_)
best_model = grid.best_estimator_
best_model
param_grid 

In [ ]:
#model evalution

In [ ]:
y_pred_final = best_model.predict(X_test_selected)
print("Final Model Accuracy:", accuracy_score(y_test, y_pred_final))
print(confusion_matrix(y_test, y_pred_final))